In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import Row
import  pyspark.sql.functions as fun
import numpy as np
from pyspark.sql.types import *

In [2]:
spark = SparkSession.builder.getOrCreate()

In [3]:
sc = spark.sparkContext

Create an RDD from a list of numbers (1,50) using numpy methods

In [47]:
li_ = np.array(range(1,51))
li_

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34,
       35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50])

Find the sum, average, maximum, minimum, and count

In [48]:
#Sum
int(li_.sum())

1275

In [56]:
#avg
int(li_.sum()) / len(li_)


25.5

In [52]:
#Count
len(li_)

50

In [45]:
#Min
int(li_.min())

1

In [46]:
#Max
int(li_.max())

49

Count how many numbers are even vs. odd.

In [66]:
print(f'count even numbers = {len(li_[li_ % 2 == 0])}')
print(f'count odd numbers ={len(li_[li_ % 2 != 0])}')  # or len(li_[li_ % 2 == 1]

count even numbers = 25
count odd numbers =25


You have the following data of people info ('Name', 'Age'), answer the following questions

In [57]:
people_data = [("Nada ", 25), ("Mona", 30), ("Ahmed", 35), ("Khaled", 40),("Ahmed", 35), ('Nada ', 25)]
rdd_people = sc.parallelize(people_data)
rdd_people.collect()

[('Nada ', 25),
 ('Mona', 30),
 ('Ahmed', 35),
 ('Khaled', 40),
 ('Ahmed', 35),
 ('Nada ', 25)]

Find the oldest person

In [76]:
old_person = rdd_people.map(lambda x: x[1]).max()
rdd_people.filter(lambda x: x[1] == old_person).collect()

[('Khaled', 40)]

Compute the average age

In [77]:
avg_age = rdd_people.map(lambda x: x[1]).sum() / rdd_people.count()
avg_age

31.666666666666668

Group all the names by their age

In [85]:
group_age = rdd_people.map(lambda x: (x[1], x[0])).groupByKey().mapValues(list)
group_age.collect()

[(30, ['Mona']),
 (40, ['Khaled']),
 (25, ['Nada ', 'Nada ']),
 (35, ['Ahmed', 'Ahmed'])]

Take the following text and put it in a text file named russia.txt and load it into rdd

"Russia is the largest country in the world by land area
Moscow is the capital city of Russia
The Russian language is one of the most widely spoken languages in the world
Russia is known for its rich history and culture
The Trans-Siberian Railway is the longest railway line in the world
Russia has a strong tradition in literature, music and ballet
The country is famous for its cold winters and vast landscapes
Russia is a major player in global energy production
"

In [88]:
rdd_russia = sc.textFile("/content/russia.txt")
rdd_russia.collect()

['Russia is the largest country in the world by land area Moscow is the capital city of Russia The Russian language is one of the most widely ',
 'spoken languages in the world Russia is known for its rich history and culture The Trans-Siberian Railway is the longest railway line in the',
 'world Russia has a strong tradition in literature, music and ballet The country is famous for its cold winters and vast landscapes Russia is a ',
 'major player in global energy production']

Count the total number of lines.



In [89]:
rdd_russia.count()

4

Count how many lines contain the word "Russia"

In [90]:
rdd_russia.filter(lambda line: "Russia" in line).count()

3

Find the most 5 frequent word in the file.

In [91]:
rdd_russia.flatMap(lambda line: line.split()).map(lambda word: (word, 1)).reduceByKey(lambda a, b: a + b).sortBy(lambda x: x[1], ascending=False).take(5)

[('is', 7), ('the', 7), ('Russia', 5), ('in', 5), ('world', 3)]

In [101]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

Tokenize words

In [104]:

rdd_file_splited = rdd_russia.flatMap(lambda line: line.split())
print(rdd_file_splited.collect())

['Russia', 'is', 'the', 'largest', 'country', 'in', 'the', 'world', 'by', 'land', 'area', 'Moscow', 'is', 'the', 'capital', 'city', 'of', 'Russia', 'The', 'Russian', 'language', 'is', 'one', 'of', 'the', 'most', 'widely', 'spoken', 'languages', 'in', 'the', 'world', 'Russia', 'is', 'known', 'for', 'its', 'rich', 'history', 'and', 'culture', 'The', 'Trans-Siberian', 'Railway', 'is', 'the', 'longest', 'railway', 'line', 'in', 'the', 'world', 'Russia', 'has', 'a', 'strong', 'tradition', 'in', 'literature,', 'music', 'and', 'ballet', 'The', 'country', 'is', 'famous', 'for', 'its', 'cold', 'winters', 'and', 'vast', 'landscapes', 'Russia', 'is', 'a', 'major', 'player', 'in', 'global', 'energy', 'production']


Remove stopwords (a, the, is, to, in, of).

In [105]:
stop_word = ['is', 'are', 'the' , 'in', 'on']
rdd_file__stop_words = rdd_russia.flatMap(lambda line: line.split()).filter(lambda word: word not in stop_word)
rdd_file__stop_words.count()

63

Count the frequency of each word

In [106]:
rdd_russia.map(lambda line: line.split()).flatMap(lambda x: x).map(lambda word: (word, 1)).reduceByKey(lambda a, b: a + b).collect()

[('Russia', 5),
 ('largest', 1),
 ('country', 2),
 ('world', 3),
 ('by', 1),
 ('land', 1),
 ('area', 1),
 ('capital', 1),
 ('of', 2),
 ('language', 1),
 ('most', 1),
 ('widely', 1),
 ('known', 1),
 ('for', 2),
 ('history', 1),
 ('and', 3),
 ('Trans-Siberian', 1),
 ('Railway', 1),
 ('line', 1),
 ('literature,', 1),
 ('music', 1),
 ('famous', 1),
 ('cold', 1),
 ('winters', 1),
 ('landscapes', 1),
 ('player', 1),
 ('energy', 1),
 ('production', 1),
 ('is', 7),
 ('the', 7),
 ('in', 5),
 ('Moscow', 1),
 ('city', 1),
 ('The', 3),
 ('Russian', 1),
 ('one', 1),
 ('spoken', 1),
 ('languages', 1),
 ('its', 2),
 ('rich', 1),
 ('culture', 1),
 ('longest', 1),
 ('railway', 1),
 ('has', 1),
 ('a', 2),
 ('strong', 1),
 ('tradition', 1),
 ('ballet', 1),
 ('vast', 1),
 ('major', 1),
 ('global', 1)]

In [111]:
schema = 'id integer, name string, age integer, salary integer'
data = [
    (1, "Ali", 25, 4000),
    (2, "Mariam", 30, 6000),
    (3, "Omar", 35, 7000),
    (4, "Sara", 28, 5000),
    (5, "Omar", 25, 6500),
    (6, "Mariam", 26, 7500)
]

df = spark.createDataFrame(data,schema)

Show schema and first 2 rows



In [114]:
df.printSchema()

df.show(2)

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- salary: integer (nullable = true)

+---+------+---+------+
| id|  name|age|salary|
+---+------+---+------+
|  1|   Ali| 25|  4000|
|  2|Mariam| 30|  6000|
+---+------+---+------+
only showing top 2 rows



Select only name and salary

In [115]:
df.select("name", "salary").show()


+------+------+
|  name|salary|
+------+------+
|   Ali|  4000|
|Mariam|  6000|
|  Omar|  7000|
|  Sara|  5000|
|  Omar|  6500|
|Mariam|  7500|
+------+------+



Find the average salary

In [116]:
df.groupBy().avg("salary").show()

+-----------+
|avg(salary)|
+-----------+
|     6000.0|
+-----------+



Filter employees older than 28

In [117]:
df.filter(df.age > 28).show()

+---+------+---+------+
| id|  name|age|salary|
+---+------+---+------+
|  2|Mariam| 30|  6000|
|  3|  Omar| 35|  7000|
+---+------+---+------+



Count distinct values in the name column

In [118]:
df.select("name").distinct().count()

4

Group by a the name column and find average salary

In [119]:
df.groupBy("name").avg("salary").show()

+------+-----------+
|  name|avg(salary)|
+------+-----------+
|  Omar|     6750.0|
|Mariam|     6750.0|
|   Ali|     4000.0|
|  Sara|     5000.0|
+------+-----------+



In [120]:
df1 = spark.read.csv("/content/NullData.csv", header=True, inferSchema=True) #this file in shared folder
df1.show()

+----+-----+-----+
|  Id| Name|Sales|
+----+-----+-----+
|emp1| John| NULL|
|emp2| NULL| NULL|
|emp3| NULL|345.0|
|emp4|Cindy|456.0|
+----+-----+-----+



Find the avg sales

In [121]:
df1.groupBy().avg("Sales").show()

+----------+
|avg(Sales)|
+----------+
|     400.5|
+----------+



Replace null name with 'Unknown' and sales with the avg sales of the column

In [126]:

total_sales = df1.select("sales").rdd.flatMap(lambda x: x).filter(lambda x: x is not None).sum()
count_sales = df1.select("sales").rdd.flatMap(lambda x: x).filter(lambda x: x is not None).count()

avg_ = total_sales / count_sales

df1_clean = df1.fillna({"name": "Unknown", "sales": avg_})

df1_clean.show()


+----+-------+-----+
|  Id|   Name|Sales|
+----+-------+-----+
|emp1|   John|400.5|
|emp2|Unknown|400.5|
|emp3|Unknown|345.0|
|emp4|  Cindy|456.0|
+----+-------+-----+

